In [5]:
import os
import time
from dotenv import load_dotenv
from dashscope.audio.tts_v2 import SpeechSynthesizer
import dashscope

# ======================
# 加载 API Key
# ======================
load_dotenv()
api_key = os.getenv("DASHSCOPE_API_KEY")

if not api_key:
    raise ValueError("❌ 未找到 DASHSCOPE_API_KEY，请检查 .env 文件")

dashscope.api_key = api_key

# ======================
# 模型与音色
# ======================
MODEL_TTS = "cosyvoice-v3-flash"
VOICE = "loongandy_v3"
SPEECH_RATE = 0.8

# ======================
# 用户输入文件名
# ======================
base_name = input("请输入文件名（无需后缀）：").strip()
input_filename = base_name + ".txt"

# ======================
# 路径
# ======================
BASE_DIR = os.getcwd()

input_txt = os.path.join(BASE_DIR, "outputs", "plaintext", input_filename)
output_mp3_dir = os.path.join(BASE_DIR, "outputs", "mp3")
os.makedirs(output_mp3_dir, exist_ok=True)

# 输出文件名：和输入一样，只是后缀 .mp3
output_filename = os.path.splitext(input_filename)[0] + ".mp3"
output_mp3 = os.path.join(output_mp3_dir, output_filename)

# ======================
# 检查输入文件
# ======================
if not os.path.isfile(input_txt):
    raise FileNotFoundError(f"❌ 找不到输入文件：{input_txt}")

print(f"▶ 正在朗读：{input_filename}")

with open(input_txt, "r", encoding="utf-8") as f:
    text = f.read().strip()

if not text:
    raise ValueError("❌ 文本文件为空")

# ======================
# TTS 合成
# ======================
synthesizer = SpeechSynthesizer(
    model=MODEL_TTS,
    voice=VOICE,
    speech_rate=SPEECH_RATE
)

start = time.time()
audio = synthesizer.call(text)
cost = time.time() - start

with open(output_mp3, "wb") as f:
    f.write(audio)

print(f"✅ 已生成：{output_mp3}")
print(f"⏱️ 耗时 {cost:.2f} 秒")

请输入文件名（无需后缀）：Gen_1_en
▶ 正在朗读：Gen_1_en.txt
✅ 已生成：/Users/Clare/dev/syncbible/outputs/mp3/Gen_1_en.mp3
⏱️ 耗时 45.82 秒


In [1]:
# 自动识别 plaintext目录下所有 Gen_*_en.txt文件的版本并转换

import os
import time
import re
from dotenv import load_dotenv
from dashscope.audio.tts_v2 import SpeechSynthesizer
import dashscope

# ======================
# 加载 API Key
# ======================
load_dotenv()
api_key = os.getenv("DASHSCOPE_API_KEY")

if not api_key:
    raise ValueError("❌ 未找到 DASHSCOPE_API_KEY，请检查 .env 文件")

dashscope.api_key = api_key

# ======================
# 模型与音色
# ======================
MODEL_TTS = "cosyvoice-v3-flash"
VOICE = "loongandy_v3"
SPEECH_RATE = 0.8

# ======================
# 路径
# ======================
BASE_DIR = os.getcwd()
PLAINTEXT_DIR = os.path.join(BASE_DIR, "outputs", "plaintext")
OUTPUT_MP3_DIR = os.path.join(BASE_DIR, "outputs", "mp3")
os.makedirs(OUTPUT_MP3_DIR, exist_ok=True)

# ======================
# 自动识别 Gen_*_en.txt
# ======================
pattern = re.compile(r"^Gen_.*_en\.txt$", re.IGNORECASE)

txt_files = [
    f for f in os.listdir(PLAINTEXT_DIR)
    if f.endswith(".txt") and pattern.match(f)
]

# 按文件名排序（避免乱序）
txt_files.sort()

if not txt_files:
    raise FileNotFoundError("❌ 未找到任何 Gen_*_en.txt 文件")

print(f"🔍 发现 {len(txt_files)} 个文本文件")
for f in txt_files:
    print("   ├─", f)

# ======================
# TTS 合成函数
# ======================
def generate_tts(input_txt, output_mp3):
    with open(input_txt, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if not text:
        raise ValueError("文本文件为空")

    synthesizer = SpeechSynthesizer(
        model=MODEL_TTS,
        voice=VOICE,
        speech_rate=SPEECH_RATE
    )

    start = time.time()
    audio = synthesizer.call(text)
    cost = time.time() - start

    with open(output_mp3, "wb") as f:
        f.write(audio)

    print(f"✅ {os.path.basename(output_mp3)}（{cost:.2f} 秒）")
    return cost


# ======================
# 批量生成
# ======================
total_start = time.time()
total_cost = 0
success_count = 0

for txt_file in txt_files:
    base_name = os.path.splitext(txt_file)[0]
    input_txt = os.path.join(PLAINTEXT_DIR, txt_file)
    output_mp3 = os.path.join(OUTPUT_MP3_DIR, base_name + ".mp3")

    try:
        cost = generate_tts(input_txt, output_mp3)
        total_cost += cost
        success_count += 1
    except Exception as e:
        print(f"❌ 失败：{txt_file} -> {e}")

# ======================
# 总结
# ======================
total_time = time.time() - total_start
print("\n📊 批量生成完成")
print(f"✅ 成功：{success_count} 个")
print(f"⏱️ 总耗时：{total_time:.2f} 秒")
print(f"⏱️ 实际合成耗时：{total_cost:.2f} 秒")

🔍 发现 50 个文本文件
   ├─ Gen_10_en.txt
   ├─ Gen_11_en.txt
   ├─ Gen_12_en.txt
   ├─ Gen_13_en.txt
   ├─ Gen_14_en.txt
   ├─ Gen_15_en.txt
   ├─ Gen_16_en.txt
   ├─ Gen_17_en.txt
   ├─ Gen_18_en.txt
   ├─ Gen_19_en.txt
   ├─ Gen_1_en.txt
   ├─ Gen_20_en.txt
   ├─ Gen_21_en.txt
   ├─ Gen_22_en.txt
   ├─ Gen_23_en.txt
   ├─ Gen_24_en.txt
   ├─ Gen_25_en.txt
   ├─ Gen_26_en.txt
   ├─ Gen_27_en.txt
   ├─ Gen_28_en.txt
   ├─ Gen_29_en.txt
   ├─ Gen_2_en.txt
   ├─ Gen_30_en.txt
   ├─ Gen_31_en.txt
   ├─ Gen_32_en.txt
   ├─ Gen_33_en.txt
   ├─ Gen_34_en.txt
   ├─ Gen_35_en.txt
   ├─ Gen_36_en.txt
   ├─ Gen_37_en.txt
   ├─ Gen_38_en.txt
   ├─ Gen_39_en.txt
   ├─ Gen_3_en.txt
   ├─ Gen_40_en.txt
   ├─ Gen_41_en.txt
   ├─ Gen_42_en.txt
   ├─ Gen_43_en.txt
   ├─ Gen_44_en.txt
   ├─ Gen_45_en.txt
   ├─ Gen_46_en.txt
   ├─ Gen_47_en.txt
   ├─ Gen_48_en.txt
   ├─ Gen_49_en.txt
   ├─ Gen_4_en.txt
   ├─ Gen_50_en.txt
   ├─ Gen_5_en.txt
   ├─ Gen_6_en.txt
   ├─ Gen_7_en.txt
   ├─ Gen_8_en.txt
   ├─ Gen_9_en